## Extract and process Somatic data from COSMIC

Search for the term 'USER' to locate codes lines where user must provide input

In [1]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

In [ ]:
start_time=time.asctime(time.localtime())
print(start_time)
#setting path to input and output files
parentpath1="USER INSERT PARENT PATH TO FOLDER WHERE YOU WILL CREATE SUB-FOLDERS PER GENE OF INTEREST AND OUTPUT EXTRACTED DATA"
os.chdir(parentpath1)
cbio_sampleids=pd.read_csv("USER INSERT PATH TO cBioPortal sample ids to remove from cosmic/df1_213_nonredundant_5_5_23.txt", sep="\t")
COSMIC_individualid=pd.read_csv("USER INSERT PATH TO COSMIC/Cosmic_Sample_Tsv_v98_GRCh38/Cosmic_Sample_v98_GRCh38.tsv", sep="\t")
COSMIC_individualid_cosmicsampleidindex=COSMIC_individualid.set_index("COSMIC_SAMPLE_ID")
#read file with folder names (40 gene names list) and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4)
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE SELECT TRANSCRIPT ID (Supplemental Table)", sep="\t")
#switch to COSMIC folder for COSMIC analysis:
parentpath2="USER INSERT PATH TO COSMIC/Cosmic_MutantCensus_Tsv_v98_GRCh38"
os.chdir(parentpath2)
COMICmasterfile= pd.read_csv("Cosmic_MutantCensus_v98_GRCh38.tsv", sep="\t", dtype={13:str,14:str})
#initializing df to collect counts to output
#FilterandQCcounts=pd.DataFrame()
cbio_sampleids_resetindex=cbio_sampleids.set_index("sampleId")

# Loop through each folder name
for index, row in TranscriptID.iterrows():
    
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #first load MANE Select/transcript identifier to a variable
    transcriptname=TranscriptID.loc[index]["MANE Select Transcript ID"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name)==False:
        os.mkdir(folder_name)
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        #make directory to store output files from code run (by date)
        if os.path.exists("Extract")==False:
            os.mkdir("Extract")
        os.chdir("Extract")
        COSMIC_gene=COMICmasterfile[COMICmasterfile["GENE_SYMBOL"].isin([folder_name])==True]
        rawcount=len(COSMIC_gene)
        print(f"raw count is: {rawcount}")
        COSMIC_gene_NaNHGVSg=COSMIC_gene[COSMIC_gene["HGVSG"].isna()==True]
        print(len(COSMIC_gene_NaNHGVSg))
        COSMIC_gene_HGVSg=COSMIC_gene.drop(COSMIC_gene_NaNHGVSg.index)
        filter1count=len(COSMIC_gene_HGVSg)
        print(f"no blank HGVSg: {filter1count}")

        COSMIC_gene_HGVSg_resetindex=COSMIC_gene_HGVSg.set_index("SAMPLE_NAME")

        #now dropping index of samples overlapping with cbio:

        cbio_to_drop=[]
        for i in cbio_sampleids_resetindex.index:
            if i in COSMIC_gene_HGVSg_resetindex.index:
                cbio_to_drop.append(i)
        
        COSMIC_gene_HGVSg_minuscbio=COSMIC_gene_HGVSg_resetindex.drop(cbio_to_drop)
        
        minuscbiocount=len(COSMIC_gene_HGVSg_minuscbio)
        print(f"minus cbio count = {minuscbiocount}")


        COSMIC_gene_HGVSg_minuscbio_cosmicsampleidindex=COSMIC_gene_HGVSg_minuscbio.reset_index().set_index("COSMIC_SAMPLE_ID")

        COSMIC_gene_HGVSg_minuscbio_individualid=COSMIC_gene_HGVSg_minuscbio_cosmicsampleidindex.join(COSMIC_individualid_cosmicsampleidindex, how='left', rsuffix='fromsampletsv')

        QC1count=len(COSMIC_gene_HGVSg_minuscbio_individualid)
        print(f"after append individual id count: {QC1count}")
        
        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples=COSMIC_gene_HGVSg_minuscbio_individualid.drop_duplicates(subset=['INDIVIDUAL_ID', 'HGVSC'])
        filter2count=len(COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples)
        print(f"after filtering duplicate indicidual id and hgvsc={filter2count}")
        
        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.fillna(0)
        #inserting column for QC result
        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.insert(1,"QCgenomiccoordinates","NaN")
        #looping over every row in dataframe
        for index, row in COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.iterrows():
            #assigning row and column indexers to respective variables to be able to call a specific position in df
            rowindex=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.index.get_loc(index)
            colindex=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.columns.get_loc('QCgenomiccoordinates')
            #length of ref allele minus 1 should match difference between stop and start (except for insertions where diff should be 1 and ref="-")
            if (int(row['GENOME_STOP'])-int(row['GENOME_START']))==(len(str(row["GENOMIC_WT_ALLELE"]))-1):
                COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.iloc[rowindex,colindex]="Pass"
            else:
                if ((int(row['GENOME_STOP'])-int(row['GENOME_START']))==1)&(row["GENOMIC_WT_ALLELE"]==0): 
                    COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.iloc[rowindex,colindex]="Pass"
                #if criteria not satisfied, variant Fails QC
                else:
                    COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples.iloc[rowindex,colindex]="Fail"
        #filter out failed variants at QC step
        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples[COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples['QCgenomiccoordinates'].str.contains("Pass")==True]
        ##print QC fail variants to new file to see why they failed (f = fail)
        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QCFAIL=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples[COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples['QCgenomiccoordinates'].str.contains("Fail")==True]

        QC2count=len(COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC)
        print(f"how many passed QC genomic coordinates={QC2count}")
        
        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC.reset_index()
        COSMIC_gene_HGVSg_only=pd.DataFrame(COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC["HGVSG"])
        COSMIC_gene_HGVSg_only_unique=COSMIC_gene_HGVSg_only.drop_duplicates(subset=["HGVSG"])
        uniquevarcount=len(COSMIC_gene_HGVSg_only_unique)
        print(f"unique HGVSg count= {uniquevarcount}")           
                     
        ####Annotate oncogenicity with OncoKB API
        json_for_oncoKB=[]
        with open('json_for_oncokb_python.json','w') as outfile:
            for index, row in  COSMIC_gene_HGVSg_only_unique.iterrows():
                #adding id to json file to identify the input variant- this identifier will be printed in output and used to match the output with input info for each variant
                json_for_oncoKB.append({"hgvsg":f"{row['HGVSG']}","id":f"{index}","referenceGenome":"GRCh38"})
            json.dump(json_for_oncoKB, outfile)
        ##querying oncoKB api
        url_oncoKBAPI="https://www.oncokb.org/api/v1/annotate/mutations/byHGVSg"
        headersnewkey = {'Authorization': 'USER ENTER API KEY HERE', 'Accept' : 'application/json', 'Content-Type' : 'application/json'}
        r_oncoKBAPI = requests.post(url_oncoKBAPI,data=open('json_for_oncokb_python.json', 'rb'), headers=headersnewkey)
        OncoKBAPI_output=pd.read_json(r_oncoKBAPI.text)
        #making new column with the input id in oncokb output file so can join back to cbio-raw-data file using index in the correct order
        OncoKBAPI_output.insert(0,"index_id","NaN")
        for index, row in OncoKBAPI_output.iterrows():
            rowindex=OncoKBAPI_output.index.get_loc(index)
            colindex=OncoKBAPI_output.columns.get_loc('index_id')
            query_identifier=row["query"]
            OncoKBAPI_output.iloc[rowindex,colindex]=query_identifier.get("id")
        #converting id from string to integer
        OncoKBAPI_output['index_id']=pd.to_numeric(OncoKBAPI_output['index_id'])
        #setting row index of the oncokb api output df to id (which is technically the row index of the input file) so now can use this common row index to match and join the 2 files
        OncoKBAPI_output_resetindex=OncoKBAPI_output.set_index('index_id')

        #joining df from OncoKB API with processed file:
        COSMIC_gene_HGVSg_only_unique_OncoKB=COSMIC_gene_HGVSg_only_unique.join(OncoKBAPI_output_resetindex)

        COSMIC_gene_HGVSg_only_unique_OncoKB_resetindex=COSMIC_gene_HGVSg_only_unique_OncoKB.set_index("HGVSG")

        COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC_resetindex=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC.set_index("HGVSG")
        COSMIC_gene_HGVSg_OncoKB=COSMIC_gene_HGVSg_minuscbio_individualid_filterduplicatesamples_QC_resetindex.join(COSMIC_gene_HGVSg_only_unique_OncoKB_resetindex)

        QC3count=len(COSMIC_gene_HGVSg_OncoKB)
        print(f"after append OncoKB, count should match QC 2 count: {QC3count}")
        
        COSMIC_gene_HGVSg_OLO=COSMIC_gene_HGVSg_OncoKB[COSMIC_gene_HGVSg_OncoKB["oncogenic"].str.contains("Likely Oncogenic|Oncogenic")==True]
        
        OLOfiltercount=(len(COSMIC_gene_HGVSg_OLO))
        print(f"O/LO filter={OLOfiltercount}")
        
        if (OLOfiltercount!=0):
            COSMIC_gene_HGVSg_OLO=COSMIC_gene_HGVSg_OLO.reset_index()
            COSMIC_gene_HGVSg_OLO["HGVSG"].to_csv("COSMIC_processed_HGVSGtoVEP.txt", sep="\t", header=None, index=False)
            QC4=pd.read_csv("COSMIC_processed_HGVSGtoVEP.txt", sep="\t", header=None)
            QC4count=len(QC4)
            print(f"HGVSg to VEP file count={QC4count}")
            COSMIC_gene_HGVSg_OLO.to_csv("COSMIC_processed.txt", sep="\t")
        else:
            QC4count=0
        output_df_counts = pd.DataFrame({'Gene': [folder_name], 'total_var_count':[rawcount], '#_has_HGVSg':[filter1count], 'minus_cbio_count':[minuscbiocount], 'individualid_append_QC_Count':[QC1count], 'filter_duplicate_samples_count':[filter2count], 'genomiccoordinates_QC_Count':[QC2count],'unique_HGVSg_count':[uniquevarcount],'oncokb_append_QC_Count':[QC3count], 'O/LO':[OLOfiltercount], 'VEP_file_QC_count':[QC4count]})
        
        FilterandQCcounts= pd.concat([FilterandQCcounts,output_df_counts]) 

        #checkpoint for where the code is while running- print gene name (directory)
        print("Current directory:", os.getcwd())
        #go back to parent directory after completing gene and start over with next gene
        os.chdir(parentpath2)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time=time.asctime(time.localtime())
print(end_time)